In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\VIREN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
# Load the dataset
# Replace 'path_to_file' with the path to the Sentiment140 dataset
data = pd.read_csv('Sentiment_analysis.csv', encoding='ISO-8859-1', header=None, usecols=[0, 5], names=['Sentiment', 'Text'])
# Map sentiment to 0 (negative) and 1 (positive)


In [5]:
data.head()

,Sentiment,Text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [6]:
data['Sentiment'] = data['Sentiment'].replace({0: 0, 4: 1})

In [7]:
data.head()

,Sentiment,Text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [8]:
data.shape

(1600000, 2)

In [9]:
data['Sentiment'].value_counts()

Sentiment
0    800000
1    800000
Name: count, dtype: int64

In [10]:
# Import necessary modules
import re
from nltk.corpus import stopwords
# Precompute stop words and compile regex patterns
stop_words = set(stopwords.words('english'))
url_pattern = re.compile(r"http\S+|www\S+|https\S+")
mention_pattern = re.compile(r'\@\w+|\#')
special_char_pattern = re.compile(r'[^\w\s]')
digit_pattern = re.compile(r'\d+')
# Optimized text preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = url_pattern.sub('', text)
    text = mention_pattern.sub('', text)
    text = special_char_pattern.sub('', text)
    text = digit_pattern.sub('', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text
# Apply preprocessing to text data
data['Text'] = data['Text'].apply(preprocess_text)

In [11]:
data.head()

,Sentiment,Text
0,0,thats bummer shoulda got david carr third day
1,0,upset cant update facebook texting might cry r...
2,0,dived many times ball managed save rest go bounds
3,0,whole body feels itchy like fire
4,0,behaving im mad cant see


In [12]:
data['Sentiment'].value_counts()

Sentiment
0    800000
1    800000
Name: count, dtype: int64

In [13]:
data.head(10)

,Sentiment,Text
0,0,thats bummer shoulda got david carr third day
1,0,upset cant update facebook texting might cry r...
2,0,dived many times ball managed save rest go bounds
3,0,whole body feels itchy like fire
4,0,behaving im mad cant see
5,0,whole crew
6,0,need hug
7,0,hey long time see yes rains bit bit lol im fin...
8,0,nope didnt
9,0,que muera


In [14]:
X_train, X_test, y_train, y_test = train_test_split(data['Text'], data['Sentiment'], test_size=0.2, random_state=42)

In [15]:
X_train.shape , X_test.shape , y_train.shape,y_test.shape

((1280000,), (320000,), (1280000,), (320000,))

In [16]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=6000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

# Fit and transform the training data, and transform the test data
X_train_tfidf = vectorizer.fit_transform(X_train)  # X_train is your training text data
X_test_tfidf = vectorizer.transform(X_test)

In [17]:
X_train_tfidf

<1280000x6000 sparse matrix of type '<class 'numpy.float64'>'
	with 7548806 stored elements in Compressed Sparse Row format>

In [ ]:
# # Train the Naive Bayes model
# nb_model = MultinomialNB()
# nb_model.fit(X_train_tfidf, y_train)

# # Make predictions
# y_pred_nb = nb_model.predict(X_test_tfidf)
# print("Naive Bayes Model Accuracy:", accuracy_score(y_test, y_pred_nb))
# print(classification_report(y_test, y_pred_nb))

Naive Bayes Model Accuracy: 0.757878125
              precision    recall  f1-score   support

           0       0.75      0.77      0.76    159494
           1       0.76      0.75      0.76    160506

    accuracy                           0.76    320000
   macro avg       0.76      0.76      0.76    320000
weighted avg       0.76      0.76      0.76    320000



In [18]:
nb_model = MultinomialNB(alpha=3.0)

# Train the model
nb_model.fit(X_train_tfidf, y_train)  # y_train is the target labels

# Predict on the test set
y_pred_nb = nb_model.predict(X_test_tfidf)

# Evaluate the model
print("Naive Bayes Model Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Model Accuracy: 0.75968125
              precision    recall  f1-score   support

           0       0.75      0.77      0.76    159494
           1       0.77      0.75      0.76    160506

    accuracy                           0.76    320000
   macro avg       0.76      0.76      0.76    320000
weighted avg       0.76      0.76      0.76    320000



In [ ]:
# Tokenize and Pad Sequences
tokenizer = Tokenizer(num_words=6000)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences to ensure consistent input length
max_len = 100
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')


Exception ignored in: <function AtomicFunction.__del__ at 0x000001649058CCC0>
Traceback (most recent call last):
  File "c:\Users\VIREN\Desktop\python\Lib\site-packages\tensorflow\python\eager\polymorphic_function\atomic_function.py", line 303, in __del__
    RUNTIME_FUNCTION_REFS.pop(key)
KeyboardInterrupt: 


In [ ]:
# Define LSTM model
lstm_model = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=max_len),
    LSTM(64),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

# Train the LSTM model
lstm_model.fit(X_train_pad, y_train, epochs=3, batch_size=128, validation_split=0.2)

# Evaluate the LSTM model
y_pred_lstm = (lstm_model.predict(X_test_pad) > 0.5).astype("int32")
print("LSTM Model Accuracy:", accuracy_score(y_test, y_pred_lstm))
print(classification_report(y_test, y_pred_lstm))


Epoch 1/3
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 767s 95ms/step - accuracy: 0.4995 - loss: 0.6933 - val_accuracy: 0.5001 - val_loss: 0.6933
Epoch 2/3
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 677s 85ms/step - accuracy: 0.5001 - loss: 0.6932 - val_accuracy: 0.4999 - val_loss: 0.6931
Epoch 3/3
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 671s 84ms/step - accuracy: 0.5001 - loss: 0.6932 - val_accuracy: 0.4999 - val_loss: 0.6932
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 125s 12ms/step
LSTM Model Accuracy: 0.50158125


c:\Users\VIREN\Desktop\python\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00    159494
           1       0.50      1.00      0.67    160506

    accuracy                           0.50    320000
   macro avg       0.25      0.50      0.33    320000
weighted avg       0.25      0.50      0.34    320000



c:\Users\VIREN\Desktop\python\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\VIREN\Desktop\python\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Define RNN model
rnn_model = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=max_len),
    SimpleRNN(64),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

# Train the RNN model
rnn_model.fit(X_train_pad, y_train, epochs=3, batch_size=128, validation_split=0.2)

# Evaluate the RNN model
y_pred_rnn = (rnn_model.predict(X_test_pad) > 0.5).astype("int32")
print("RNN Model Accuracy:", accuracy_score(y_test, y_pred_rnn))
print(classification_report(y_test, y_pred_rnn))


c:\Users\VIREN\Desktop\python\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/3
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 326s 40ms/step - accuracy: 0.7030 - loss: 0.5771 - val_accuracy: 0.4999 - val_loss: 0.6932
Epoch 2/3
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 334s 42ms/step - accuracy: 0.5776 - loss: 0.6434 - val_accuracy: 0.7699 - val_loss: 0.4866
Epoch 3/3
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 352s 44ms/step - accuracy: 0.7697 - loss: 0.4875 - val_accuracy: 0.7793 - val_loss: 0.4698
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 68s 7ms/step
RNN Model Accuracy: 0.7787125
              precision    recall  f1-score   support

           0       0.78      0.77      0.78    159494
           1       0.78      0.78      0.78    160506

    accuracy                           0.78    320000
   macro avg       0.78      0.78      0.78    320000
weighted avg       0.78      0.78      0.78    320000



In [ ]:
# Save the model in .h5 format
# model.save('sentiment_model.h5')
# Save the trained RNN model
rnn_model.save('rnn_sentiment_model.h5')

In [20]:
import joblib
# Save the model as a pickle file
# joblib.dump(nb_model, 'sentiment_model.pkl')
joblib.dump(nb_model, 'naive_bayes_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

['vectorizer.pkl']

In [ ]:
import joblib
# Save the tokenizer
joblib.dump(tokenizer, 'tokenizer.pkl')

['tokenizer.pkl']